# Frequent Pattern Mining Dashboard

Interactive dashboard for mining frequent patterns using the Apriori algorithm with adjustable support thresholds.

In [ ]:
# Import required libraries
import os
import tempfile
import warnings
from itertools import combinations
from collections import defaultdict, Counter
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output, FileLink
from mlxtend.frequent_patterns import apriori, association_rules
from google.colab import files

# Suppress warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load and preprocess transactions
def load_transactions(filepath):
    """Load transactions from CSV file"""
    transactions = []
    with open(filepath, 'r') as f:
        for line in f:
            # Split by comma, then split items by space
            parts = line.strip().split(',')
            if len(parts) >= 2:
                transaction_id = parts[0]
                items = parts[1].strip().split()
                transactions.append(items)
    return transactions

# Load the dataset
transactions = load_transactions('transactions.csv')
print(f"Loaded {len(transactions)} transactions")
print(f"Sample transactions: {transactions[:15]}")

Loaded 15 transactions
Sample transactions: [['Milk', 'Bread', 'Egg'], ['Bread', 'Butter'], ['Milk', 'Bread', 'Butter', 'Egg'], ['Beer', 'Chips'], ['Milk', 'Bread', 'Butter'], ['Bread', 'Egg'], ['Milk', 'Diaper', 'Beer', 'Chips'], ['Bread', 'Diaper', 'Beer', 'Cola'], ['Milk', 'Bread', 'Diaper', 'Beer'], ['Bread', 'Butter', 'Egg'], ['Milk', 'Egg'], ['Milk', 'Bread', 'Butter', 'Chips'], ['Bread', 'Butter'], ['Milk', 'Diaper', 'Beer'], ['Milk', 'Bread', 'Chips']]


In [ ]:
# Build one-hot encoded DataFrame for mlxtend

# Collect all unique items
all_items = sorted({item for tx in transactions for item in tx})

# Create one-hot rows
rows = []
for tx in transactions:
    row = {item: (item in tx) for item in all_items}
    rows.append(row)

# One-hot DataFrame
df_onehot = pd.DataFrame(rows)
print(f"One-hot DataFrame shape: {df_onehot.shape}")
print(f"Columns (items): {list(df_onehot.columns)}")

One-hot DataFrame shape: (15, 8)
Columns (items): ['Beer', 'Bread', 'Butter', 'Chips', 'Cola', 'Diaper', 'Egg', 'Milk']


In [ ]:
# Enhanced Apriori Dashboard

# Initialize cache and signature function
_cached_results = {}

def _data_signature(df):
    return (id(df), df.shape)

def run_apriori_cached(df_onehot, min_support, max_len=None, use_colnames=True):
    sig = (_data_signature(df_onehot), float(min_support), int(max_len) if max_len else None)
    if sig in _cached_results:
        return _cached_results[sig]
    freq = apriori(df_onehot, min_support=min_support, use_colnames=use_colnames, max_len=max_len)
    # ensure support numeric
    freq['support'] = freq['support'].astype(float)
    _cached_results[sig] = freq
    return freq

# ---------- Utility: format itemsets ----------
def format_itemset(itemset):
    if isinstance(itemset, (frozenset, set, list, tuple)):
        return ', '.join(sorted(itemset))
    return str(itemset)

# ---------- UI widgets ----------
six_quick_buttons_values = ['0.05', '0.1', '0.2', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.9']

support_slider = widgets.FloatSlider(
    value=0.2, min=0.01, max=1.0, step=0.01,
    description='<b>Min Support:</b>', continuous_update=False,
    readout_format='.2f', layout=widgets.Layout(width='420px')
)

# Quick buttons with different light backgrounds
quick_buttons_values = six_quick_buttons_values # Using the full list provided in the notebook
quick_buttons_styles = ['info', 'success', 'warning', 'primary', 'danger'] # Different light styles
quick_buttons_widgets = []
for i, val in enumerate(quick_buttons_values):
    btn = widgets.Button(
        description=val,
        tooltip=f'{float(val)*100:.0f}%',
        layout=widgets.Layout(width='60px'),
        button_style=quick_buttons_styles[i % len(quick_buttons_styles)] # Cycle through styles
    )
    quick_buttons_widgets.append(btn)

quick_buttons = widgets.HBox(quick_buttons_widgets)
quick_access_label = widgets.HTML('<b>Quick access:</b>')

max_len_selector = widgets.IntRangeSlider(
    value=[1, 3], min=1, max=max(3, df_onehot.shape[1]), step=1,
    description='<b>Itemset size (min,max):</b>', continuous_update=False, layout=widgets.Layout(width='520px')
)

top_k_int = widgets.IntSlider(value=10, min=3, max=50, step=1, description='<b>Top K:</b>', layout=widgets.Layout(width='300px'))

rule_conf_slider = widgets.FloatSlider(value=0.6, min=0.0, max=1.0, step=0.01, description='<b>Min Confidence:</b>', continuous_update=False)
rule_lift_slider = widgets.FloatSlider(value=1.0, min=0.01, max=10.0, step=0.01, description='<b>Min Lift:</b>', continuous_update=False)

mine_button = widgets.Button(description='🔍 Mine', button_style='primary', layout=widgets.Layout(width='120px'))
export_itemsets_btn = widgets.Button(description='Export Itemsets (CSV)', layout=widgets.Layout(width='180px'))
export_rules_btn = widgets.Button(description='Export Rules (CSV)', layout=widgets.Layout(width='180px'))

output_area = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px'))

# Quick button handlers
def _set_support_factory(val):
    def _inner(b):
        support_slider.value = float(val)
    return _inner

for btn in quick_buttons.children:
    btn.on_click(_set_support_factory(btn.description))

# ---------- Mining & Display logic ----------
def display_results(min_support, min_size, max_size, top_k, min_conf, min_lift):
    with output_area:
        clear_output(wait=True)
        display(HTML("<h2 style='color:#007bff; text-align:center;'>🔎 Apriori Results</h2>")) # Enhanced header color
        total_tx = len(df_onehot)
        display(HTML(f"<p style='color:#333;'><strong>Total transactions:</strong> {total_tx} &nbsp; • &nbsp; <strong>Min support:</strong> {min_support:.3f} ({int(min_support*total_tx)} tx)</p>"))

        # run apriori (use max_len for efficiency)
        freq_df = run_apriori_cached(df_onehot, min_support=min_support, max_len=max_size, use_colnames=True)
        # filter by size range
        freq_df['size'] = freq_df['itemsets'].apply(lambda s: len(s))
        freq_filtered = freq_df[(freq_df['size'] >= min_size) & (freq_df['size'] <= max_size)].copy()

        if freq_filtered.empty:
            display(HTML("<div style='padding:12px; border-left:4px solid #ff9800; background:#fff3e0; color:#e65100;'><strong>⚠️ No frequent itemsets found at this support & size range.</strong><br>Try lowering support or expanding itemset size range.</div>")) # Warning color
        else:
            # sort and show top K (by support)
            freq_sorted = freq_filtered.sort_values('support', ascending=False).reset_index(drop=True)
            top_k = min(top_k, len(freq_sorted))
            display(HTML(f"<h4 style='color:#007bff;'>Top {top_k} itemsets (by support)</h4>")) # Sub-header color

            display_df = freq_sorted.head(top_k).copy()
            display_df['itemset'] = display_df['itemsets'].apply(format_itemset)
            display_df['count'] = (display_df['support'] * total_tx).round().astype(int)
            display_display_df = display_df[['itemset','size','support','count']].rename(columns={'support':'support (frac)'})
            display(display_display_df.style.format({'support (frac)': '{:.3f}'}).set_table_attributes("style='width:80%;'"))

            # Bar chart: top_k support
            fig, ax = plt.subplots(figsize=(8, 5))
            # Using a more attractive color palette
            colors = plt.cm.viridis(np.linspace(0, 1, top_k))[::-1]
            ax.barh(range(top_k), display_df.head(top_k)['support'][::-1], color=colors)
            ax.set_yticks(range(top_k))
            ax.set_yticklabels(display_df.head(top_k)['itemset'][::-1])
            ax.invert_yaxis()
            ax.set_xlabel('Support (fraction of transactions)')
            ax.set_title(f'Top {top_k} Frequent Itemsets', color='#007bff')
            plt.tight_layout()
            display(fig)
            plt.close(fig)

        # Association rules
        rules = pd.DataFrame()
        if not freq_filtered.empty:
            all_freq_for_rules = run_apriori_cached(df_onehot, min_support=min_support, use_colnames=True)
            rules = association_rules(all_freq_for_rules, metric="confidence", min_threshold=min_conf)
            # filter by lift and by sizes
            rules['antecedent_len'] = rules['antecedents'].apply(lambda x: len(x))
            rules['consequent_len'] = rules['consequents'].apply(lambda x: len(x))
            rules = rules[(rules['antecedent_len'] + rules['consequent_len']).between(min_size+0, max_size)]
            rules = rules[rules['lift'] >= min_lift]
            if rules.empty:
                display(HTML("<div style='padding:12px; border-left:4px solid #fbc02d; background:#fff8e1;'><strong>ℹ️ No association rules meet the confidence/lift thresholds.</strong></div>"))
            else:
                display(HTML(f"<h4 style='color:#007bff;'>Association Rules (confidence \u2265 {min_conf:.2f}, lift \u2265 {min_lift:.2f})</h4>")) # Sub-header color
                rules_display = rules.copy()
                rules_display['antecedents'] = rules_display['antecedents'].apply(format_itemset)
                rules_display['consequents'] = rules_display['consequents'].apply(format_itemset)
                rules_display['support_count'] = (rules_display['support'] * total_tx).round().astype(int)
                display_cols = ['antecedents','consequents','support','support_count','confidence','lift']
                display(rules_display[display_cols].sort_values(['confidence','lift'], ascending=False).reset_index(drop=True).style.format({'support':'{:.3f}','confidence':'{:.3f}','lift':'{:.3f}'}))

                # Scatter plot: support vs confidence (size = lift)
                fig2, ax2 = plt.subplots(figsize=(8,6))
                # Using a more vibrant colormap for scatter points
                sc = ax2.scatter(rules['support'], rules['confidence'], s=(rules['lift']*20).clip(10,200), c=rules['lift'], cmap='plasma', alpha=0.8)
                ax2.set_xlabel('Support')
                ax2.set_ylabel('Confidence')
                ax2.set_title('Rules: support vs confidence (marker size ~ lift)', color='#007bff')
                plt.colorbar(sc, label='Lift')
                # annotate top rules by confidence
                top_rules = rules.sort_values(['confidence','lift'], ascending=False).head(5)
                for _, r in top_rules.iterrows():
                    ax2.annotate(format_itemset(r['antecedents']) + ' \u2192 ' + format_itemset(r['consequents']),
                                 (r['support'], r['confidence']), xytext=(10,10), textcoords='offset points', fontsize=7, color='#333')
                plt.tight_layout()
                display(fig2)
                plt.close(fig2)

        # store last results for export
        global _last_freq_df, _last_rules_df
        _last_freq_df = freq_filtered.copy() if not freq_filtered.empty else pd.DataFrame()
        _last_rules_df = rules.copy() if isinstance(rules, pd.DataFrame) else pd.DataFrame()

# ---------- Export handlers ----------
def _export_df_to_csv(df, filename_prefix='export'):
    if df.empty:
        with output_area:
            display(HTML("<div style='padding:10px; border-left:4px solid #ef5350; background:#ffebee;'>No data to export.</div>"))
        return
    tmpdir = tempfile.gettempdir()
    fname = os.path.join(tmpdir, f"{filename_prefix}_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv")
    # Convert any frozensets/sets to strings
    df_to_save = df.copy()
    for col in df_to_save.columns:
        if df_to_save[col].apply(lambda x: isinstance(x, (frozenset, set, list, tuple))).any():
            df_to_save[col] = df_to_save[col].apply(format_itemset)
    df_to_save.to_csv(fname, index=False)
    with output_area:
        display(HTML(f"<div style='padding:10px; border-left:4px solid #28a745; background:#d4edda; color:#155724;'>Exported CSV: <strong>{fname}</strong></div>")) # Success color
        files.download(fname)

def on_export_itemsets(b):
    _export_df_to_csv(_last_freq_df, filename_prefix='frequent_itemsets')

def on_export_rules(b):
    _export_df_to_csv(_last_rules_df, filename_prefix='association_rules')

export_itemsets_btn.on_click(on_export_itemsets)
export_rules_btn.on_click(on_export_rules)

# ---------- Main interaction ----------
def on_mine_clicked(b):
    min_support = support_slider.value
    min_size, max_size = max(1, max_len_selector.value[0]), max_len_selector.value[1]
    display_results(min_support=min_support,
                    min_size=min_size,
                    max_size=max_size,
                    top_k=top_k_int.value,
                    min_conf=rule_conf_slider.value,
                    min_lift=rule_lift_slider.value)

mine_button.on_click(on_mine_clicked)

# ---------- Layout assemble ----------
controls_col1 = widgets.VBox([support_slider, quick_access_label, quick_buttons, max_len_selector, top_k_int, mine_button])
controls_col2 = widgets.VBox([widgets.Label("Association Rules Filters (post-mining):"), rule_conf_slider, rule_lift_slider, widgets.HBox([export_itemsets_btn, export_rules_btn])])
controls = widgets.HBox([controls_col1, controls_col2], layout=widgets.Layout(justify_content='space-between'))

header = widgets.HTML("""
    <div style='font-family:sans-serif'>
      <h1 style='color:#0056b3; margin:0; text-align:center; padding-bottom:10px;'>Frequent Pattern Mining Dashboard</h1>
      <p style='margin-top:4px; color:#555; margin-bottom:12px; text-align:center;'>
        Uses <strong>Apriori</strong> for itemset mining and <strong>association_rules</strong> for rule mining.
        Adjust support and size, then click <strong>Mine</strong>. Export CSVs for figures/tables in your report.
      </p>
    </div>
""")

dashboard = widgets.VBox([header, controls, output_area], layout=widgets.Layout(width='100%'))

display(dashboard)

# Optionally run initial mining with default widgets
on_mine_clicked(None)